In [1]:
import os
from pathlib import Path
#import sys

dirpath_base = Path(r'D:\WORK\Salvador\repo\A1_OUinp')
print(dirpath_base)
#sys.path.append(str(dirpath_base))

os.chdir(dirpath_base)

from create_base_cfg import create_base_cfg
from create_net_params import create_net_params

D:\WORK\Salvador\repo\A1_OUinp


libNeuroML >>> WARNING - Please install optional dependencies to use SED-ML features:
libNeuroML >>> WARNING - pip install pyneuroml[combine]


In [2]:
cfg = create_base_cfg()
net_params = create_net_params(cfg)

dirpath_out = dirpath_base / 'revision' / 'ouinp'
cfg.save(str(dirpath_out / 'cfg.json'))
net_params.save(str(dirpath_out / 'netParams.json'))

Saving simConfig to D:\WORK\Salvador\repo\A1_OUinp\revision\ouinp\cfg.json ... 
Saving netParams to D:\WORK\Salvador\repo\A1_OUinp\revision\ouinp\netParams.json ... 


In [5]:
import json
import numpy as np
from deepdiff import DeepDiff

vers = ['ouinp', 'samn']

# Load configs
cfg, netpar = {}, {}
for ver in vers:
    with open(dirpath_base / 'revision' / ver / 'cfg.json', 'rb') as fid:
        cfg[ver] = json.load(fid)
    with open(dirpath_base / 'revision' / ver / 'netParams.json', 'rb') as fid:
        netpar[ver] = json.load(fid)

# Comapare cfg and netParams between ouinp and samn
cfg_diff = DeepDiff(cfg['ouinp'], cfg['samn'],
                ignore_order=True, report_repetition=True)
netpar_diff = DeepDiff(netpar['ouinp'], netpar['samn'],
                ignore_order=True, report_repetition=True)

for diff in [cfg_diff, netpar_diff]:
    diff['dictionary_item_added'] = list(diff['dictionary_item_added'])
    diff['dictionary_item_removed'] = list(diff['dictionary_item_removed'])

# Save the diffs
with open(dirpath_base / 'revision' / 'cfg_diff.json', 'w') as fid:
    json.dump(cfg_diff, fid, indent=4, default=str)
with open(dirpath_base / 'revision' / 'netpar_diff.json', 'w') as fid:
    json.dump(netpar_diff, fid, indent=4, default=str)

In [5]:
import pickle as pkl
import xarray as xr

# Read conn.pkl of both setups
conn_data = {}
for name in ['ouinp', 'samn']:
    with open(dirpath_base / 'revision' / name / 'conn.pkl', 'rb') as fid:
        conn_data[name] = pkl.load(fid)

mats = {'pkl': ['pmat', 'lmat', 'wmat'],
        'cfg': ['wmat']}

# Check that conn['pmat'] contains the same keys in both setups
for mat in mats['pkl']:
    pops_ = [list(conn_data[name][mat].keys()) for name in ['ouinp', 'samn']]
    if set(pops_[0]) == set(pops_[1]):
        print(f"{mat} pops contain the same elements.")
    else:
        print(f"{mat} pops differ.")

# Convert mats to xarrays
pops = list(conn_data['samn']['pmat'].keys())
npops = len(pops)
mats_data = {}
for ver in vers:   # ouinp, samn
    mats_data[ver] = {}
    for src, mats_ in mats.items():   # pkl, cfg
        mats_data[ver][src] = {}
        for m in mats_:   # wmat, lmat, pmat
            M = xr.DataArray(
                np.zeros((npops, npops)), 
                coords=[pops, pops], dims=['pre', 'post'])
            # Source
            if src == 'pkl':
                S = conn_data[ver][m]
            else:
                S = cfg[ver]['simConfig'][m]
            # Copy
            for pre in pops:
                for post in pops:
                    if (pre in S) and (post in S[pre]):
                        M.loc[pre, post] = S[pre][post]
            mats_data[ver][src][m] = M

pmat pops contain the same elements.
lmat pops contain the same elements.
wmat pops contain the same elements.


In [8]:
# Compare mats between the setups
for src, mats_ in mats.items():   # pkl, cfg
    for m in mats_:   # wmat, lmat, pmat
        M1 = mats_data['ouinp'][src][m]
        M2 = mats_data['samn'][src][m]
        b = np.all((M1 == M2).values.ravel())
        print(f'{m} ({src}) equal in both versions: {b}')

# Compare wmat between cfg and pkl for each setup
for ver in vers:
    M1 = mats_data[ver]['cfg']['wmat']
    M2 = mats_data[ver]['pkl']['wmat']
    b = np.all((M1 == M2).values.ravel())
    print(f'wmat ({ver}) is equal in cfg and pkl: {b}')

pmat (pkl) equal in both versions: True
lmat (pkl) equal in both versions: True
wmat (pkl) equal in both versions: True
wmat (cfg) equal in both versions: False
wmat (ouinp) is equal in cfg and pkl: True
wmat (samn) is equal in cfg and pkl: False
